# Random Forest Fertilizer-Deficit Trainer (I-NEW-GEN)
รันทีละ cell จากบนลงล่าง (Runtime → Run all ก็ได้) ใช้เวลารวมประมาณ 1 นาที

1. **Setup & Download** – โหลดไลบรารีและ Open-Source Dataset
2. **Agronomic Model** – สูตรปฐพีวิทยาไม่เป็นเชิงเส้น + เกณฑ์กรมวิชาการเกษตร (2565)
3. **Build Dataset** – จับคู่ดินจริงกับ 9 โปรไฟล์พืช
4. **Train & Evaluate** – 80:20 hold-out + 5-Fold CV
5. **Save & Download** – ได้ `agronomic_rf_dataset.csv`, `rf_fertilizer_model.joblib`, `rf_evaluation_report.json`

> Target คำนวณจากสูตร ไม่ใช่ค่าวัดภาคสนาม R² สูงจึงหมายถึงโมเดลเรียนรู้สูตรได้

## 1. Setup & Download

In [ ]:
import json, time, urllib.request
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

OUT_DIR = Path("rf_output"); OUT_DIR.mkdir(exist_ok=True)
RAW_URL = "https://raw.githubusercontent.com/Gladiator07/Harvestify/master/Data-processed/crop_recommendation.csv"
RAW_CSV = OUT_DIR / "Crop_recommendation.csv"

urllib.request.urlretrieve(RAW_URL, RAW_CSV)
raw = pd.read_csv(RAW_CSV)
print("scikit-learn", sklearn.__version__, "| rows:", len(raw), "| crops:", raw["label"].nunique())
raw.head()

## 2. Agronomic model & crop profiles

In [ ]:
P_TO_P2O5_FACTOR = 2.29
K_TO_K2O_FACTOR = 1.20
FEATURE_NAMES = ["crop_profile_id", "root_depth_m", "bulk_density_g_cm3", "soil_n_mg_kg", "soil_p_mg_kg",
                 "soil_k_mg_kg", "ph_value", "moisture_pct", "exg_green_pct"]
TARGETS = ["deficit_n_g_m2", "deficit_p2o5_g_m2", "deficit_k2o_g_m2"]

# (crop, stage, root_depth_m, target_N, target_P2O5, target_K2O [g/m2], ph_min, ph_max)  -- DOA 2565 / ARDA
PROFILES = [
    ("maize", "1", 0.15, 2.50, 2.00, 1.50, 5.5, 7.0), ("maize", "2", 0.25, 6.00, 1.50, 2.50, 5.5, 7.0),
    ("maize", "3", 0.30, 2.75, 1.50, 2.88, 5.5, 7.0),
    ("groundnut", "1", 0.15, 1.00, 0.20, 0.40, 5.5, 6.8), ("groundnut", "2", 0.20, 1.50, 0.20, 0.50, 5.5, 6.8),
    ("groundnut", "3", 0.25, 0.50, 0.16, 0.54, 5.5, 6.8),
    ("coffee", "1", 0.20, 3.00, 0.40, 2.00, 5.0, 6.2), ("coffee", "2", 0.30, 8.00, 0.56, 4.00, 5.0, 6.2),
    ("coffee", "3", 0.35, 4.50, 0.60, 11.00, 5.0, 6.2),
]
STAGE_EXG_MEAN = {1: 35.0, 2: 60.0, 3: 55.0}
BULK_DENSITIES = [1.25, 1.35, 1.45]
KG_HA_PER_MG_KG = 2.0  # 15 cm layer ~ 2e6 kg/ha -> 1 mg/kg ~ 2 kg/ha
N_ESTIMATORS, MAX_DEPTH, MIN_LEAF, SEED = 120, 12, 2, 42


def agronomic_targets(profile_id, root_depth_m, target_n, target_p2o5, target_k2o, ph_min, ph_max,
                      bulk_density, soil_n, soil_p, soil_k, ph_val, moisture_pct, exg_pct):
    """Non-linear nutrient deficits (g/m2): pH lock-up, moisture diffusion, ExG foliar N demand."""
    if ph_val < ph_min:
        ph_penalty = max(0.60, 1.0 - 0.18 * (ph_min - ph_val))
    elif ph_val > ph_max:
        ph_penalty = max(0.65, 1.0 - 0.15 * (ph_val - ph_max))
    else:
        ph_penalty = 1.0
    if moisture_pct < 25.0:
        moisture_factor = max(0.70, 0.70 + 0.30 * (moisture_pct / 25.0))
    elif moisture_pct > 80.0:
        moisture_factor = max(0.75, 1.0 - 0.01 * (moisture_pct - 80.0))
    else:
        moisture_factor = 1.0
    expected_exg = 35.0 if profile_id % 3 == 1 else (60.0 if profile_id % 3 == 2 else 55.0)
    canopy_n = float(np.clip(1.0 + 0.0025 * (expected_exg - exg_pct), 0.92, 1.12))

    mass = bulk_density * root_depth_m
    m_n = soil_n * mass * 0.30 * ph_penalty * moisture_factor
    m_p = soil_p * mass * P_TO_P2O5_FACTOR * 0.20 * ph_penalty ** 1.3 * moisture_factor
    m_k = soil_k * mass * K_TO_K2O_FACTOR * 0.15 * ph_penalty * moisture_factor
    return (max(0.0, target_n * canopy_n - m_n), max(0.0, target_p2o5 - m_p), max(0.0, target_k2o - m_k))

## 3. Build dataset (2,200 real soil states × 9 profiles)

In [ ]:
def build_dataset(raw):
    rng = np.random.default_rng(SEED)
    rows = []
    for pid, (_, _, root, t_n, t_p, t_k, ph_min, ph_max) in enumerate(PROFILES, start=1):
        for src_idx, r in raw.iterrows():
            bd = float(rng.choice(BULK_DENSITIES))
            exg = float(np.clip(rng.normal(STAGE_EXG_MEAN[(pid - 1) % 3 + 1], 12.0), 5.0, 90.0))
            s_n, s_p, s_k = r["N"] / KG_HA_PER_MG_KG, r["P"] / KG_HA_PER_MG_KG, r["K"] / KG_HA_PER_MG_KG
            ph = float(np.clip(r["ph"], 4.0, 8.0))
            moist = float(np.clip(r["humidity"], 10.0, 95.0))
            dn, dp, dk = agronomic_targets(pid, root, t_n, t_p, t_k, ph_min, ph_max, bd, s_n, s_p, s_k, ph, moist, exg)
            rows.append({"source_row_id": int(src_idx), "source_label": r["label"], "crop_profile_id": pid,
                         "root_depth_m": root, "bulk_density_g_cm3": bd, "soil_n_mg_kg": round(s_n, 3),
                         "soil_p_mg_kg": round(s_p, 3), "soil_k_mg_kg": round(s_k, 3), "ph_value": round(ph, 3),
                         "moisture_pct": round(moist, 3), "exg_green_pct": round(exg, 3),
                         "deficit_n_g_m2": round(dn, 5), "deficit_p2o5_g_m2": round(dp, 5),
                         "deficit_k2o_g_m2": round(dk, 5)})
    return pd.DataFrame(rows)

df = build_dataset(raw)
df.to_csv(OUT_DIR / "agronomic_rf_dataset.csv", index=False)
print(df.shape)
df.head()

## 4. Train & Evaluate (กดรันเทรนจริงที่ cell นี้)

In [ ]:
def metrics(y_true, y_pred):
    out = {}
    for i, name in enumerate(["N", "P2O5", "K2O"]):
        out[name] = {"r2": float(r2_score(y_true[:, i], y_pred[:, i])),
                     "rmse": float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))),
                     "mae": float(mean_absolute_error(y_true[:, i], y_pred[:, i]))}
    out["overall"] = {"r2": float(r2_score(y_true, y_pred)),
                      "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
                      "mae": float(mean_absolute_error(y_true, y_pred))}
    return out

def new_rf():
    # Standardise targets: P2O5 is ~10x smaller than N/K and would be ignored by the multi-output MSE split.
    rf = RandomForestRegressor(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, min_samples_leaf=MIN_LEAF,
                               random_state=SEED, n_jobs=-1)
    return TransformedTargetRegressor(regressor=rf, transformer=StandardScaler())

t0 = time.time()
x, y, groups = df[FEATURE_NAMES].to_numpy(), df[TARGETS].to_numpy(), df["source_row_id"].to_numpy()
# Group by source row so one real soil sample never appears in both train and test.
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED).split(x, y, groups))
rf = new_rf().fit(x[tr], y[tr])
holdout = metrics(y[te], rf.predict(x[te]))

folds = [metrics(y[b], new_rf().fit(x[a], y[a]).predict(x[b]))
         for a, b in GroupKFold(n_splits=5).split(x, y, groups)]
cv = {k: {m: {"mean": float(np.mean([f[k][m] for f in folds])), "std": float(np.std([f[k][m] for f in folds]))}
          for m in ["r2", "rmse", "mae"]} for k in ["N", "P2O5", "K2O", "overall"]}
imp = {n: round(float(v) * 100, 2) for n, v in zip(FEATURE_NAMES, rf.regressor_.feature_importances_)}

report = {
    "model": "RandomForestRegressor (scikit-learn)",
    "hyperparameters": {"n_estimators": N_ESTIMATORS, "max_depth": MAX_DEPTH, "min_samples_leaf": MIN_LEAF, "random_state": SEED},
    "dataset": {"source_url": RAW_URL, "open_rows_used": int(df["source_row_id"].nunique()),
                "total_samples": int(len(df)), "train_samples": int(len(tr)), "test_samples": int(len(te)),
                "pairing_note": "All 2,200 real soil states are paired with each of the 9 crop profiles; one random bulk density per pair.",
                "targets_note": "Targets are computed by the agronomic non-linear formula, not field measurements."},
    "holdout_80_20": holdout, "cross_validation_5fold": cv, "feature_importance_pct": imp,
    "training_seconds": round(time.time() - t0, 1),
}

summary = pd.DataFrame({k: {"R2 hold-out": holdout[k]["r2"], "RMSE": holdout[k]["rmse"], "MAE": holdout[k]["mae"],
                            "R2 5-fold CV": cv[k]["r2"]["mean"]} for k in holdout}).T.round(4)
display(summary)
display(pd.Series(imp, name="Feature importance (%)").sort_values(ascending=False).to_frame())
print("training seconds:", report["training_seconds"])

## 5. Save & Download

In [ ]:
joblib.dump({"model": rf, "report": report}, OUT_DIR / "rf_fertilizer_model.joblib")
(OUT_DIR / "rf_evaluation_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

# Sanity check: reload and predict (maize stage 2, N=15, P=5, K=60 mg/kg)
m = joblib.load(OUT_DIR / "rf_fertilizer_model.joblib")["model"]
print("pred g/m2 [N, P2O5, K2O]:", m.predict([[2, 0.25, 1.35, 15, 5, 60, 6.0, 45, 55]])[0].round(3))

try:
    from google.colab import files
    import shutil
    shutil.make_archive("rf_output", "zip", OUT_DIR)
    files.download("rf_output.zip")
except ImportError:
    print("Not on Colab - files are in", OUT_DIR.resolve())

นำไฟล์ `.joblib`, `.csv`, `.json` ไปวางที่ `app/data/` ของโปรเจกต์ (ต้องใช้ scikit-learn เวอร์ชันใกล้เคียงกับที่พิมพ์ใน cell 1 มิฉะนั้นอาจโหลดโมเดลไม่ได้)